<a href="https://colab.research.google.com/github/somboro08/Hlang/blob/main/mini_gpt_bariba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-GPT Bariba ↔ Français — à partir de zéro (Colab)

Ce notebook prend le tokenizer et les datasets tokenisés déjà préparés
(`tokenizer.json`, `lm_*.bin`, `*_tokenized_*.jsonl`) et construit un petit
GPT en PyTorch : couche d'embedding, embedding de position, blocs
transformer, tête de sortie.

**Avant de lancer :** uploade dans le même dossier (ou dans Google Drive
puis adapte les chemins) tous les fichiers du dossier `tokenizer_and_data/`
livré avec ce notebook :
- `tokenizer.json`
- `lm_train.bin`, `lm_valid.bin`, `lm_test.bin`
- `translation_tokenized_{train,valid,test}.jsonl`
- `dictionary_tokenized_{train,valid,test}.jsonl`


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/bariba_gpt_data')

Mounted at /content/drive


In [2]:
import os

file_path = "tokenizer.json"
if os.path.exists(file_path):
    print(f"Le fichier '{file_path}' est bien présent dans le répertoire courant : {os.getcwd()}")
else:
    print(f"Le fichier '{file_path}' n'a PAS été trouvé dans le répertoire courant : {os.getcwd()}")
    print("Veuillez vous assurer qu'il est bien présent dans '/content/drive/MyDrive/bariba_gpt_data' ou que le chemin d'accès est correct.")

Le fichier 'tokenizer.json' est bien présent dans le répertoire courant : /content/drive/MyDrive/bariba_gpt_data


In [3]:
!pip install -q tokenizers
import numpy as np, json, math, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cuda


## 1. Charger le tokenizer

C'est un tokenizer BPE byte-level entraîné spécifiquement sur ton corpus
bariba-français (vocab_size = 8000). Byte-level = aucun caractère bariba
(ɛ, ɔ, ɓ, ɗ, ŋ, ã, ĩ, tons...) ne peut être "inconnu" : tout se décompose
en octets si besoin, donc jamais de perte d'information.


In [4]:
tok = Tokenizer.from_file("tokenizer.json")
VOCAB_SIZE = tok.get_vocab_size()
PAD_ID = tok.token_to_id("<pad>")
BOS_ID = tok.token_to_id("<bos>")
EOS_ID = tok.token_to_id("<eos>")
BAR_ID = tok.token_to_id("<bariba>")
FR_ID  = tok.token_to_id("<fr>")
SEP_ID = tok.token_to_id("<sep>")
print("vocab_size:", VOCAB_SIZE)

# petit test
enc = tok.encode("Yè ba gberu da, bibu ba bɔ̃ɔ binun nɔni baakia.")
print(enc.ids)
print(tok.decode(enc.ids))


vocab_size: 8000
[659, 350, 1294, 429, 18, 1308, 350, 534, 269, 263, 3333, 288, 1172, 662, 763, 20]
Yè ba gberu da, bibu ba bɔ̃ɔ binun nɔni baakia.


## 2. Le modèle — GPT minimal

**C'est ici que se trouve l'embedding** : `nn.Embedding(vocab_size, d_model)`
transforme chaque identifiant de token en un vecteur de dimension `d_model`,
appris pendant l'entraînement (pas précalculé). On ajoute un embedding de
position, puis des blocs transformer standards (auto-attention causale +
MLP), puis une tête linéaire vers le vocabulaire.


In [11]:
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(),
            nn.Linear(4 * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=512, n_head=8, n_layer=6, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.position_embedding(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

## 3. Pré-entraînement type "langage" (next-token prediction)

Utilise `lm_train.bin` / `lm_valid.bin` : c'est tout le corpus (définitions
+ exemples, bariba et français mélangés) concaténé en une seule longue
séquence de tokens, au format nanoGPT (tableau uint16 sur disque).


In [12]:
BLOCK_SIZE = 256

train_data = np.memmap('lm_train.bin', dtype=np.uint16, mode='r')
valid_data = np.memmap('lm_valid.bin', dtype=np.uint16, mode='r')

def get_batch(data, batch_size=32, block_size=BLOCK_SIZE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

model = MiniGPT(VOCAB_SIZE, BLOCK_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(2000):
    xb, yb = get_batch(train_data)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 200 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch(valid_data)
            _, vloss = model(xv, yv)
        model.train()
        print(f"step {step}: train loss {loss.item():.3f}  valid loss {vloss.item():.3f}")

step 0: train loss 9.192  valid loss 8.025
step 200: train loss 4.356  valid loss 4.481
step 400: train loss 3.758  valid loss 4.124
step 600: train loss 3.124  valid loss 3.852
step 800: train loss 2.667  valid loss 3.654
step 1000: train loss 2.315  valid loss 3.709
step 1200: train loss 1.719  valid loss 3.625
step 1400: train loss 1.348  valid loss 3.809
step 1600: train loss 0.930  valid loss 3.967
step 1800: train loss 0.686  valid loss 4.070


## 4. Fine-tuning supervisé : traduction bariba ↔ français

Utilise `translation_tokenized_{train,valid,test}.jsonl` : chaque exemple a
`input_ids` (prompt + réponse) et `labels` (avec `-100` sur le prompt, donc
la perte ne compte que sur la partie à générer — traduction, ici).
Même mécanisme pour `dictionary_tokenized_*.jsonl` (définition d'un mot).


In [7]:
def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8')]

def collate(batch, pad_id=PAD_ID, max_len=BLOCK_SIZE):
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, ex in enumerate(batch):
        ids = ex['input_ids'][:max_len]
        lab = ex['labels'][:max_len]
        input_ids[i, :len(ids)] = torch.tensor(ids)
        labels[i, :len(lab)]    = torch.tensor(lab)
    return input_ids.to(device), labels.to(device)

train_pairs = load_jsonl('translation_tokenized_train.jsonl')
valid_pairs = load_jsonl('translation_tokenized_valid.jsonl')

import random
def get_finetune_batch(pairs, batch_size=32):
    batch = random.sample(pairs, batch_size)
    return collate(batch)

for step in range(1000):
    xb, yb = get_finetune_batch(train_pairs)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_finetune_batch(valid_pairs)
            _, vloss = model(xv, yv)
        model.train()
        print(f"step {step}: train loss {loss.item():.3f}  valid loss {vloss.item():.3f}")


step 0: train loss 9.861  valid loss 8.088
step 100: train loss 1.559  valid loss 1.476
step 200: train loss 0.830  valid loss 0.706
step 300: train loss 0.432  valid loss 0.525
step 400: train loss 0.295  valid loss 0.415
step 500: train loss 0.196  valid loss 0.364
step 600: train loss 0.158  valid loss 0.249
step 700: train loss 0.133  valid loss 0.252
step 800: train loss 0.145  valid loss 0.302
step 900: train loss 0.062  valid loss 0.221


## 5. Générer une traduction

Génération gloutonne simple (greedy) à partir d'un prompt formaté comme
pendant l'entraînement : `<bos> <bariba> ... <sep> <fr>`.


In [13]:
@torch.no_grad()
def translate(text, src_lang="bariba", max_new_tokens=40, beam_width=3):
    src_id, tgt_id = (BAR_ID, FR_ID) if src_lang == "bariba" else (FR_ID, BAR_ID)
    initial_ids = [BOS_ID, src_id] + tok.encode(text).ids + [SEP_ID, tgt_id]

    # Initialize beam: list of (score, sequence_ids)
    # scores are log probabilities, higher is better
    beam = [(0.0, torch.tensor([initial_ids], dtype=torch.long, device=device))]

    model.eval()
    for _ in range(max_new_tokens):
        all_candidates = []
        for score, ids in beam:
            if ids[0, -1].item() == EOS_ID: # If sequence already ended, keep it
                all_candidates.append((score, ids))
                continue

            logits, _ = model(ids[:, -BLOCK_SIZE:])
            # Get log probabilities for the next token
            log_probs = F.log_softmax(logits[0, -1], dim=-1)

            # Get top 'beam_width' candidates for the next token
            top_log_probs, top_indices = torch.topk(log_probs, beam_width)

            for i in range(beam_width):
                next_token_log_prob = top_log_probs[i].item()
                next_token_id = top_indices[i].item()
                new_ids = torch.cat([ids, torch.tensor([[next_token_id]], device=device)], dim=1)
                new_score = score + next_token_log_prob
                all_candidates.append((new_score, new_ids))

        # Select the top 'beam_width' candidates from all possibilities
        beam = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_width]

        # If all sequences in the beam have ended, stop early
        if all(ids[0, -1].item() == EOS_ID for score, ids in beam):
            break

    # Pick the best sequence from the beam (highest score)
    best_score, best_ids = beam[0]
    return tok.decode(best_ids[0].tolist())

print(translate("Sunɔn alufawa u gura tɔbirimɔ.", src_lang="bariba", beam_width=5))

Sunɔn alufawa u gura tɔbirimɔ. le musulman au mari.


## Où sont les "vecteurs" (embeddings) ?

Après entraînement, la matrice apprise `model.token_embedding.weight` est
de taille `(vocab_size, d_model)` : une ligne = le vecteur appris pour un
token. On peut l'extraire à tout moment :

```python
embedding_matrix = model.token_embedding.weight.detach().cpu().numpy()
# embedding_matrix.shape == (8000, 256)
```

C'est ça, "l'embedding du dataset" — mais il n'existe qu'**après** avoir
entraîné le modèle, pas avant. Avant l'entraînement, tout ce qu'on peut
préparer, c'est le tokenizer et les identifiants (ce que ce notebook fait).
